# Mutual Information

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 05.03 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/05_Information_Theory/04_mutual_information.ipynb)

---

## 🎯 Learning Objective

Define mutual information $I(X;Y)$ as the reduction in uncertainty about one variable given another, and state the data processing inequality.

---

## 📐 Theory

KL divergence (`05.02`) compares two distributions over the *same* variable. **Mutual
information** asks a related but distinct question about *two different* random variables: how
much does observing one tell you about the other?

### Definition: mutual information as an expected KL divergence

For jointly distributed discrete random variables $X$ and $Y$ with joint PMF $P(x,y)$ and
marginals $P(x), P(y)$ (recall joint/marginal distributions from
[`03.04`](../03_Probability_and_Statistics/04_joint_conditional_marginal.ipynb)), the **mutual
information** is

$$I(X;Y) = \sum_{x,y} P(x,y) \log_2\frac{P(x,y)}{P(x)P(y)} = D_{KL}\big(P(x,y) \,\|\, P(x)P(y)\big)$$

This is literally the KL divergence between the true joint distribution and the joint
distribution you'd get if $X$ and $Y$ were independent ($P(x)P(y)$). That framing makes an
important fact obvious: $I(X;Y) = 0$ exactly when $X\perp Y$ (independence), and by Gibbs'
inequality (`05.02`) $I(X;Y) \ge 0$ always — mutual information can never be negative, unlike
correlation, which can be zero even when variables are strongly (nonlinearly) dependent, while
mutual information is zero *only* under true independence.

### Mutual information as reduction in uncertainty

An equivalent, often more intuitive formula connects $I(X;Y)$ directly to entropy. Define the
**conditional entropy** $H(Y|X) = \sum_x P(x) H(Y|X=x)$ — the average remaining uncertainty
about $Y$ once you know $X$. Then

$$I(X;Y) = H(Y) - H(Y|X) = H(X) - H(X|Y)$$

Read the first form: $I(X;Y)$ is *how much $X$ reduces your uncertainty about $Y$*, measured in
the same bits as entropy. If knowing $X$ tells you nothing about $Y$, $H(Y|X)=H(Y)$ and
$I(X;Y)=0$. If knowing $X$ pins $Y$ down completely, $H(Y|X)=0$ and $I(X;Y)=H(Y)$ — the maximum
possible, since you can never reduce uncertainty about $Y$ by more than $Y$'s own entropy. The
symmetry $I(X;Y)=I(Y;X)$ (unlike KL divergence, mutual information *is* symmetric) follows from
expanding both forms via Bayes' rule (`03.05`) — it doesn't matter which variable you call "the
one you already know."

### The data processing inequality

Suppose $X, Y, Z$ form a **Markov chain**, written $X \to Y \to Z$, meaning $Z$ depends on $X$
*only through* $Y$ (formally, $Z$ is conditionally independent of $X$ given $Y$) — think of $Y$
as a processed or compressed version of $X$, and $Z$ as something computed from $Y$ alone,
without further access to $X$. The **data processing inequality** states:

$$I(X;Z) \le I(X;Y)$$

Processing data can only destroy information about the original source, never create it. Each
additional stage in a pipeline that only has access to the previous stage's output is a
one-way valve: information about $X$ can leak out at every step but never flow back in. This is
precisely why, in the visualization below, mutual information about $X$ shrinks monotonically as
a signal passes through successive noisy stages of a chain.

### Sufficient statistics

A statistic $T(X)$ (some function of the data, like a sample mean) is called **sufficient** for
a parameter $\theta$ if it captures *all* the information $X$ carries about $\theta$: formally,
$I(\theta; X) = I(\theta; T(X))$ — compressing $X$ down to $T(X)$ loses nothing. The data
processing inequality guarantees $I(\theta;T(X)) \le I(\theta;X)$ automatically (since
$\theta \to X \to T(X)$ is a Markov chain); sufficiency is exactly the special case where that
inequality becomes equality — the compression happened to be lossless *for the purpose of
learning $\theta$*, even though $T(X)$ is a much smaller object than $X$ itself. This is the
same "compress without losing what matters" idea that motivates representation learning, which
is where mutual information reconnects with deep learning below.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Passing a signal through a chain of increasingly noisy channels lets us watch mutual
information about the original source decay monotonically -- the data processing inequality
made visible.

In [ ]:
def joint_pmf(x, y, n_x, n_y):
    table = np.zeros((n_x, n_y))
    for a, b in zip(x, y):
        table[a, b] += 1
    return table / table.sum()

def mutual_info_bits(joint):
    """I(X;Y) = sum p(x,y) log2(p(x,y) / (p(x)p(y))), summing only over the joint's support --
    matches the D_KL(P(x,y) || P(x)P(y)) definition exactly."""
    p_x = joint.sum(axis=1, keepdims=True)
    p_y = joint.sum(axis=0, keepdims=True)
    support = joint > 0
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = joint / (p_x * p_y)
    return np.sum(joint[support] * np.log2(ratio[support]))

def noisy_relabel(source, keep_prob, n_categories, rng):
    """With probability keep_prob, copy the input category through unchanged; otherwise
    replace it with a fresh uniformly random category -- a simple noisy channel."""
    keep_mask = rng.random(len(source)) < keep_prob
    return np.where(keep_mask, source, rng.integers(0, n_categories, size=len(source)))

rng = np.random.default_rng(0)
n_samples, n_categories = 20_000, 4
X = rng.integers(0, n_categories, size=n_samples)

# Build a 4-stage Markov chain X -> Y -> Z -> W, each stage only noisily copying the previous one
Y = noisy_relabel(X, 0.8, n_categories, rng)
Z = noisy_relabel(Y, 0.8, rng=rng, n_categories=n_categories)
W = noisy_relabel(Z, 0.8, rng=rng, n_categories=n_categories)

stages = {"X (source)": X, "Y = f(X)": Y, "Z = f(Y)": Z, "W = f(Z)": W}
I_with_X = [mutual_info_bits(joint_pmf(X, stage, n_categories, n_categories)) for stage in stages.values()]

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(range(len(stages)), I_with_X, marker="o", color="#4C72B0", lw=2, markersize=8)
ax.set_xticks(range(len(stages)))
ax.set_xticklabels(stages.keys())
ax.set_ylabel("I(X ; stage)  [bits]")
ax.set_title("Data processing inequality: information about X\ncan only shrink along a Markov chain")
for i, val in enumerate(I_with_X):
    ax.annotate(f"{val:.3f}", (i, val), textcoords="offset points", xytext=(0, 8), ha="center")
plt.tight_layout()
plt.show()

print("Mutual information with the ORIGINAL source X, at each stage of the chain:")
for name, val in zip(stages.keys(), I_with_X):
    print(f"  I(X; {name.split(' ')[0]}) = {val:.4f} bits")
is_monotonic = all(I_with_X[i] >= I_with_X[i + 1] - 1e-9 for i in range(len(I_with_X) - 1))
print(f"\nMonotonically non-increasing along the chain: {is_monotonic} (the data processing inequality)")

## 🐍 Implementation from Scratch

We implement mutual information directly from the joint distribution, verify it against
`sklearn.metrics.mutual_info_score`, confirm $I(X;X)=H(X)$ and $I(X;Y_{\text{indep}})\approx 0$
as boundary cases, then confirm the data processing inequality numerically along a longer
Markov chain.

In [ ]:
from sklearn.metrics import mutual_info_score

def entropy_bits(p, eps=1e-12):
    p = np.clip(p, eps, 1.0)
    return -np.sum(p * np.log2(p))

# --- Cross-check our mutual_info_bits against sklearn (which reports NATS, not bits) ---
rng = np.random.default_rng(1)
n_categories = 4
X = rng.integers(0, n_categories, size=20_000)
Y_noisy = noisy_relabel(X, keep_prob=0.6, n_categories=n_categories, rng=rng)

joint_noisy = joint_pmf(X, Y_noisy, n_categories, n_categories)
I_ours = mutual_info_bits(joint_noisy)
I_sklearn_bits = mutual_info_score(X, Y_noisy) / np.log(2)   # sklearn returns nats; convert to bits
print(f"I(X;Y) at keep_prob=0.6:  ours = {I_ours:.4f} bits   sklearn (converted) = {I_sklearn_bits:.4f} bits")
print(f"match: {np.isclose(I_ours, I_sklearn_bits, atol=1e-3)}")

# --- Boundary cases: I(X;X) should equal H(X) exactly; I(X;independent) should be ~0 ---
joint_self = joint_pmf(X, X, n_categories, n_categories)
H_X = entropy_bits(joint_self.sum(axis=1))
I_self = mutual_info_bits(joint_self)
print(f"\nI(X;X) = {I_self:.4f} bits   H(X) = {H_X:.4f} bits   match: {np.isclose(I_self, H_X)}")

Y_independent = rng.integers(0, n_categories, size=len(X))   # genuinely unrelated to X
I_indep = mutual_info_bits(joint_pmf(X, Y_independent, n_categories, n_categories))
print(f"I(X;Y_independent) = {I_indep:.4f} bits  (near 0, as theory predicts for true independence)")

# --- Sweep the noisy channel's keep_prob from 0 (pure noise) to 1 (perfect copy) and
# confirm I(X;Y) rises monotonically from ~0 to H(X) -- the full range mutual information can take ---
print(f"\n{'keep_prob':>10} | {'I(X;Y) bits':>12}")
for keep_prob in [0.0, 0.25, 0.5, 0.75, 0.95, 1.0]:
    Y = noisy_relabel(X, keep_prob, n_categories, rng)
    I_val = mutual_info_bits(joint_pmf(X, Y, n_categories, n_categories))
    print(f"{keep_prob:10.2f} | {I_val:12.4f}")
print(f"(H(X) = {H_X:.4f} bits is the ceiling -- reached exactly when keep_prob=1, i.e. Y=X)")

## 🤖 Why This Matters for AI

Mutual information formalizes what a good learned representation should do: keep the
information relevant to a downstream task while discarding everything else. **Feature
selection** methods (`sklearn.feature_selection.mutual_info_classif`) rank input features by
$I(\text{feature}; \text{label})$ directly. **Contrastive representation learning** — the
technique behind CLIP, SimCLR, and modern embedding models — trains an encoder using the
**InfoNCE** loss, which is explicitly designed as a tractable lower bound on mutual information
between two augmented views (or matched modalities) of the same underlying data. We implement
InfoNCE from scratch below and show that minimizing it drives up an embedding's *retrieval
accuracy* — a direct, measurable proxy for how much mutual information the learned embeddings
actually preserve. The data processing inequality also has a sobering implication for deep
networks: each layer can only lose information about the raw input relative to the layer before
it — the entire premise behind `05.05`'s information bottleneck theory of generalization.

In [ ]:
# InfoNCE: the contrastive loss behind CLIP, SimCLR, and modern embedding models.
# It's a tractable LOWER BOUND on I(anchor; positive) -- minimizing it maximizes a proxy for
# the mutual information between two views/modalities of the same underlying signal.
def infonce_loss_and_accuracy(anchors, positives, temperature=0.1):
    # Normalize to unit vectors so the dot product is exactly cosine similarity
    anchors = anchors / np.linalg.norm(anchors, axis=1, keepdims=True)
    positives = positives / np.linalg.norm(positives, axis=1, keepdims=True)
    similarities = anchors @ positives.T / temperature   # (batch, batch): row i = anchor i vs every positive
    similarities -= similarities.max(axis=1, keepdims=True)   # log-sum-exp stability (Module 06 covers this trick)
    exp_sims = np.exp(similarities)
    probs = exp_sims / exp_sims.sum(axis=1, keepdims=True)
    batch_size = anchors.shape[0]
    # The InfoNCE loss is cross-entropy (05.03) against a "batch classification" task: for each
    # anchor, correctly pick its TRUE positive out of all batch_size candidates
    loss = -np.mean(np.log(np.clip(np.diag(probs), 1e-12, 1.0)))
    retrieval_accuracy = np.mean(np.argmax(probs, axis=1) == np.arange(batch_size))
    return loss, retrieval_accuracy

rng = np.random.default_rng(0)
dim, batch_size = 16, 64

# Case 1: a "well-trained" encoder -- each anchor/positive pair shares a common signal, with
# only small independent noise distinguishing the two views (like two augmentations of one image)
shared_signal = rng.normal(size=(batch_size, dim))
anchors_trained = shared_signal + rng.normal(scale=0.1, size=(batch_size, dim))
positives_trained = shared_signal + rng.normal(scale=0.1, size=(batch_size, dim))
loss_trained, acc_trained = infonce_loss_and_accuracy(anchors_trained, positives_trained)

# Case 2: an "untrained" encoder -- anchors and positives carry NO shared information at all
anchors_untrained = rng.normal(size=(batch_size, dim))
positives_untrained = rng.normal(size=(batch_size, dim))
loss_untrained, acc_untrained = infonce_loss_and_accuracy(anchors_untrained, positives_untrained)

chance_accuracy = 1 / batch_size
print(f"{'':22s} {'InfoNCE loss':>14} {'retrieval acc':>15}")
print(f"{'well-trained encoder':22s} {loss_trained:14.4f} {acc_trained:15.3f}")
print(f"{'untrained encoder':22s} {loss_untrained:14.4f} {acc_untrained:15.3f}")
print(f"{'chance level':22s} {'--':>14} {chance_accuracy:15.3f}")
print(f"\nWith batch_size={batch_size}, retrieval accuracy directly measures how well the embedding")
print(f"can pick out the TRUE positive among {batch_size} candidates -- a low InfoNCE loss and high")
print(f"retrieval accuracy both indicate the embeddings preserve mutual information between views,")
print(f"exactly the quantity I(X;Y) this notebook defined, now optimized as a training objective.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Two binary variables — $X$ (rain / no rain) and $Y$ (umbrella / no umbrella) — have joint PMF
   $P(X{=}\text{no rain}, Y{=}\text{no umbrella})=0.45$, $P(\text{no rain},\text{umbrella})=0.05$,
   $P(\text{rain},\text{no umbrella})=0.10$, $P(\text{rain},\text{umbrella})=0.40$. Confirm the
   joint sums to 1, compute the marginals $P(X), P(Y)$ by hand, then compute $I(X;Y)$ term by
   term using $\sum_{x,y}P(x,y)\log_2\frac{P(x,y)}{P(x)P(y)}$. As a sanity check, confirm your
   result is non-negative and verify it with `mutual_info_bits`.

<details>
<summary>💡 Solution</summary>

The joint sums to $0.45+0.05+0.10+0.40=1.00$. Marginals: $P(X{=}\text{no rain})=0.50$,
$P(X{=}\text{rain})=0.50$, $P(Y{=}\text{no umbrella})=0.55$, $P(Y{=}\text{umbrella})=0.45$.
The four terms are $0.45\log_2\frac{0.45}{0.5\times0.55}\approx 0.3197$,
$0.05\log_2\frac{0.05}{0.5\times0.45}\approx -0.1085$,
$0.10\log_2\frac{0.10}{0.5\times0.55}\approx -0.1459$, and
$0.40\log_2\frac{0.40}{0.5\times0.45}\approx 0.3320$, summing to $I(X;Y)\approx 0.3973$ bits
— positive, as Gibbs' inequality requires for any non-independent pair. `mutual_info_bits`
applied to this joint PMF array returns the same `0.3973` bits.

</details>

### 💭 UNDERSTAND
2. Let $X$ be uniform on $\{-1, 0, 1\}$ and $Y=X^2$ (so $Y=1$ when $X=\pm1$, $Y=0$ when $X=0$).
   Predict, using only the Theory section's contrast between mutual information and correlation:
   is the Pearson correlation between $X$ and $Y$ closer to $0$ or $1$? Is $I(X;Y)$ closer to $0$
   or to $H(Y)$? Then simulate both to check.

<details>
<summary>💡 Solution</summary>

Correlation is (near) exactly $0$: $Y$ is symmetric in $X$ (it only depends on $|X|$), so
positive and negative deviations of $X$ contribute with opposite sign to
$\text{Cov}(X,Y)=\mathbb{E}[XY]-\mathbb{E}[X]\mathbb{E}[Y]$, canceling out — a simulation with
200,000 samples gives correlation $\approx 0.001$, indistinguishable from $0$ up to sampling
noise. Yet $I(X;Y)$ equals $H(Y)$ *exactly* (not just "closer to" it): $Y$ is a deterministic,
non-constant function of $X$, so knowing $X$ removes *all* remaining uncertainty about $Y$ —
$H(Y|X)=0$ — which by $I(X;Y)=H(Y)-H(Y|X)$ forces $I(X;Y)=H(Y)\approx 0.918$ bits. This is the
Theory section's warning made concrete: correlation is blind to this strong, purely nonlinear
dependence, while mutual information detects it fully.

</details>

### ✏️ DERIVE
3. Derive the identity $I(X;Y) = H(X) + H(Y) - H(X,Y)$ starting from the definition
   $I(X;Y)=\sum_{x,y}P(x,y)\log_2\frac{P(x,y)}{P(x)P(y)}$, where
   $H(X,Y)=-\sum_{x,y}P(x,y)\log_2 P(x,y)$ is the joint entropy. (Hint: split the log of the
   ratio into three separate sums.)

<details>
<summary>💡 Solution outline</summary>

Split the log:
$$I(X;Y) = \sum_{x,y}P(x,y)\log_2 P(x,y) - \sum_{x,y}P(x,y)\log_2 P(x) - \sum_{x,y}P(x,y)\log_2 P(y)$$
The first sum is exactly $-H(X,Y)$ by definition. For the second sum, $\log_2 P(x)$ doesn't
depend on $y$, so $\sum_{x,y}P(x,y)\log_2 P(x) = \sum_x \log_2 P(x)\sum_y P(x,y) = \sum_x
P(x)\log_2 P(x) = -H(X)$ (using $\sum_y P(x,y)=P(x)$, the marginalization identity from
`03.04`). By the identical argument, the third sum equals $-H(Y)$. Substituting:
$$I(X;Y) = -H(X,Y) - (-H(X)) - (-H(Y)) = H(X) + H(Y) - H(X,Y)$$
which is the claimed identity. A numerical check across several random joint PMFs confirms both
sides agree to floating-point precision, and this identity gives a second, independent way to
read $I(X;Y)$: as *how much smaller* the joint entropy is than the sum of the two marginal
entropies — dependence between $X$ and $Y$ is exactly what makes $H(X,Y) < H(X)+H(Y)$.

</details>

### 🐍 IMPLEMENT
4. Using `noisy_relabel`, build a Markov chain $X\to Y\to Z$ where $Y$ has `keep_prob=0.9` and
   $Z$ has `keep_prob=0.3` applied to $Y$. Compute $I(X;Y)$, $I(Y;Z)$, and $I(X;Z)$, and confirm
   $I(X;Z) \le \min(I(X;Y), I(Y;Z))$ — a stronger form of the data processing inequality than
   the notebook's simple monotonicity check.

<details>
<summary>✅ How to know you're right</summary>

With `keep_prob=0.9` for $X\to Y$ and `keep_prob=0.3` for $Y\to Z$ (and, say, 50,000 samples
over a 4-category alphabet), a representative run gives $I(X;Y)\approx 1.50$ bits,
$I(Y;Z)\approx 0.17$ bits, and $I(X;Z)\approx 0.14$ bits — and indeed
$I(X;Z) \le \min(I(X;Y), I(Y;Z)) = 0.17$ holds. The inequality should hold for any `keep_prob`
choice you try, not just this one — if you ever find $I(X;Z)$ exceeding both $I(X;Y)$ and
$I(Y;Z)$, that signals a bug in your joint-PMF construction (most likely a sample-size-driven
estimation issue at small `n_samples`, since mutual information estimated from finite samples is
always non-negatively biased), not a real violation of the theorem.

</details>

### 🤖 APPLY
5. A churn-prediction model has three candidate binary features to select from: feature A is
   strongly predictive of the churn label, feature B is weakly predictive, and feature C is pure
   noise independent of the label. Simulate all three (e.g. by copying the label with different
   flip probabilities for A and B, and drawing C independently), and rank them by
   $I(\text{feature};\text{label})$ using `mutual_info_bits`. Confirm your ranking matches
   `sklearn.feature_selection.mutual_info_classif`'s ranking (or `mutual_info_score`, converted
   from nats to bits) on the same data.

<details>
<summary>✅ What you should observe</summary>

The ranking should recover A > B > C by mutual information, with C landing at (or extremely
close to) $0$ bits — a representative run gives $I(A;\text{label})\approx 0.39$ bits,
$I(B;\text{label})\approx 0.01$ bits, and $I(C;\text{label})\approx 0.00$ bits. `sklearn`'s
mutual information function (after converting its nats output to bits by dividing by $\ln 2$)
should agree with your `mutual_info_bits` values to within a small numerical tolerance, since
both are computing the exact same discrete quantity from empirical frequencies. This is feature
selection in miniature — real feature-selection pipelines rank dozens or hundreds of candidate
features by exactly this score before ever training a model.

</details>

### 🚀 CHALLENGE
6. In the AI section, vary `temperature` in `infonce_loss_and_accuracy` across
   $\{0.01, 0.1, 0.5, 2.0\}$ for the well-trained embeddings, and plot both loss and retrieval
   accuracy against temperature. Explain what happens to the gradient signal at very low
   temperature (hint: look at how peaked `probs` becomes) and connect your finding to why
   contrastive learning papers treat temperature as a sensitive hyperparameter rather than an
   arbitrary constant.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer's sweep shows retrieval accuracy staying at (or very near) $1.0$ across the
entire range for well-separated embeddings, while the loss itself varies enormously (from
near-$0$ at $T=0.01$ to several nats at $T=2.0$) — accuracy alone doesn't reveal what's happening
to the *gradient*. Inspecting `probs` directly shows that at low temperature, each anchor's
distribution over candidates becomes extremely peaked (near-one-hot, with the true positive
receiving probability essentially $1$ and every negative essentially $0$), which means the loss
is already near its minimum and its gradient with respect to the embeddings is correspondingly
tiny — there's almost no signal left to push the embeddings further apart, even for pairs that
are only *barely* correctly ranked. At high temperature, probabilities flatten out (entropy of
each row approaches its ceiling), which keeps the gradient alive but also weakens the loss's
ability to sharply distinguish a correct match from a near-miss. A strong answer connects this to
why CLIP, SimCLR, and related papers report tuning temperature carefully (or learning it as a
parameter) rather than fixing it arbitrarily: it directly controls the tradeoff between "gradient
signal strength" and "how sharply the loss penalizes near-miss negatives," a tension invisible if
you only look at final accuracy.

</details>

---

## 📚 Further Reading
- Cover & Thomas, *Elements of Information Theory*, Ch. 2.8–2.9 (Mutual Information, Data Processing Inequality)
- Oord, Li & Vinyals, ["Representation Learning with Contrastive Predictive Coding"](https://arxiv.org/abs/1807.03748) (InfoNCE)
- Radford et al., ["Learning Transferable Visual Models From Natural Language Supervision"](https://arxiv.org/abs/2103.00020) (CLIP)

---

*Next notebook: [The Information Bottleneck](05_information_bottleneck.ipynb)*